In [13]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [14]:
builder = (SparkSession.builder
           .appName("connect-kafka-streaming")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark-catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder, ['org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1']).getOrCreate()

In [15]:
df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka:9092")
      .option("subscribe", "users")
      .option("startingOffsets", "earliest")
      .load())

In [16]:
schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('gender', StringType(), True),
    StructField('country', StringType(), True)])

df = df.withColumn('value', from_json(col('value').cast("STRING"), schema))

In [17]:
df = df.select(
    col('value.id').alias('id'),
    col('value.name').alias('name'),
    col('value.age').alias('age'),
    col('value.gender').alias('gender'),
    col('value.country').alias('country'))

In [18]:
df = (df.select('age', 'country', 'gender').filter("age >= 21").groupBy('country', 'gender').agg(avg('age').alias('Average age')).orderBy('country', 'gender'))

In [19]:
query = (df.writeStream
         .outputMode('complete')
         .format('console')
         .start())

25/06/01 10:45:21 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-39a7ab06-d04b-4476-a651-366785b8cb0f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/06/01 10:45:21 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/06/01 10:45:21 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+------+------------------+
|  country|gender|       Average age|
+---------+------+------------------+
|Australia|     F| 43.26923076923077|
|Australia|     M| 43.78947368421053|
|   Brazil|     F| 38.04545454545455|
|   Brazil|     M| 43.68181818181818|
|   Canada|     F|  38.1764705882353|
|   Canada|     M|              38.5|
|    China|     F|             42.05|
|    China|     M| 39.05263157894737|
|  Germany|     F|              38.5|
|  Germany|     M|              41.0|
|    India|     F| 49.03703703703704|
|    India|     M|              44.7|
|  Moldova|     F|              44.0|
|  Moldova|     M|              45.0|
|       UK|     F|              43.0|
|       UK|     M|  37.8235294117647|
|      USA|     F| 39.15384615384615|
|      USA|     M|41.411764705882355|
+---------+------+------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+---------+------+------------------+
|  country|gender|       Average age|
+---------+------+------------------+
|Australia|     F| 43.26923076923077|
|Australia|     M|             42.75|
|   Brazil|     F| 38.04545454545455|
|   Brazil|     M| 43.68181818181818|
|   Canada|     F|  38.1764705882353|
|   Canada|     M|              38.5|
|    China|     F|             42.05|
|    China|     M| 39.05263157894737|
|  Germany|     F|              38.5|
|  Germany|     M|              41.0|
|    India|     F| 49.03703703703704|
|    India|     M|              44.7|
|  Moldova|     F|              44.0|
|  Moldova|     M|              45.0|
|       UK|     F|              43.0|
|       UK|     M|  37.8235294117647|
|      USA|     F| 39.15384615384615|
|      USA|     M|41.411764705882355|
+---------+------+------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+---------+------+------------------+
|  country|gender|       Average age|
+---------+------+------------------+
|Australia|     F| 43.26923076923077|
|Australia|     M|             42.75|
|   Brazil|     F| 38.04545454545455|
|   Brazil|     M| 43.68181818181818|
|   Canada|     F|  38.1764705882353|
|   Canada|     M|              38.5|
|    China|     F|             42.05|
|    China|     M| 39.05263157894737|
|  Germany|     F|              38.5|
|  Germany|     M|              41.0|
|    India|     F| 49.03703703703704|
|    India|     M|              44.7|
|  Moldova|     F|              44.0|
|  Moldova|     M|              45.0|
|       UK|     F|              43.0|
|       UK|     M| 36.94444444444444|
|      USA|     F| 39.15384615384615|
|      USA|     M|41.411764705882355|
+---------+------+------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+---------+------+------------------+
|  country|gender|       Average age|
+---------+------+------------------+
|Australia|     F| 43.26923076923077|
|Australia|     M|             42.75|
|   Brazil|     F| 38.04545454545455|
|   Brazil|     M| 43.68181818181818|
|   Canada|     F|  38.1764705882353|
|   Canada|     M|              38.5|
|    China|     F|             42.05|
|    China|     M| 39.05263157894737|
|  Germany|     F|              38.5|
|  Germany|     M|              41.0|
|    India|     F| 49.03703703703704|
|    India|     M|              44.7|
|  Moldova|     F|              44.0|
|  Moldova|     M|              45.0|
|       UK|     F|            43.375|
|       UK|     M| 36.94444444444444|
|      USA|     F| 39.15384615384615|
|      USA|     M|41.411764705882355|
+---------+------+------------------+



In [20]:
query.stop()

In [21]:
spark.stop()